<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [15]</a>'.</span>

<link rel="stylesheet" href="notebooks/styles.css">

<div class="title-wrap">
  <h1 class="title-main" style="font-weight: bold; font-size: 2.65rem; margin-bottom: 0.5rem;">
  Spatial Data Science Approaches to Wildfire Severity Modeling
</h1>
<h2 class="title-sub" style="font-style: italic; font-size: 1.8rem; margin-top: 0rem; margin-bottom: 0.2rem;">
  A GIS‑Driven, Tree‑Based Machine Learning Analysis of California Wildfires
</h2>
</div>

# Module 8B: *Fire Ignition Feature Ablation*
##### Version Number: 4.0
---
### Contents  
> 1. **
> 2. **
> 3. **
---
### Notes
---
### Inputs
---
### Outputs 
---
### User Defined Dependencies

In [1]:
import os
import sys

# Allow import of custom modules from the parent directory
sys.path.append(os.path.abspath(os.path.join('..')))

from src.data_utils import *
from src.model_utils import *
from src.plot_utils import *

---
### Third Party Dependencies

In [2]:
# Core libraries
import pandas as pd
import numpy as np
import json

# Modeling libraries
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
from sklearn.model_selection import train_test_split

---

###  Load Data

In [3]:
X_ignition = pd.read_csv('../data/processed/X_ignition.csv')
y_ignition = pd.read_csv('../data/processed/y_ignition.csv').squeeze()  # Load as Series      
details_ignition = pd.read_csv('../data/processed/details_ignition.csv')

pal_details = pd.read_csv('../data/processed/pal_details.csv')
pal_X = pd.read_csv('../data/processed/pal_X.csv')
pal_y = pd.read_csv('../data/processed/pal_y.csv')

best_strategy = pd.read_csv('../data/processed/ignition_best_strategy.csv')

with open('model_parameters_ignition.json', 'r') as f:
    model_parameters = json.load(f)

with open('feature_sets.json', 'r') as f:
    feature_sets = json.load(f)

## Subset

In [4]:
reform = pd.concat([X_ignition,y_ignition], axis=1)
subset = subset_df(reform, 'Target_Ignition', 500)

y = subset['Target_Ignition']
X = subset.drop(columns='Target_Ignition')

In [5]:
X_rf, y_rf = apply_balancing('RF', best_strategy, X, y)
X_xgb, y_xgb = apply_balancing('XGB', best_strategy, X, y)

## Build Models

In [6]:
RF_parameters = model_parameters['Random Forest']
XGB_parameters = model_parameters['XGBoost']

# Build tuned models
ignition_xbg = xgb.XGBClassifier(**XGB_parameters)
ignition_rf = RandomForestClassifier(**RF_parameters)

display(RF_parameters)
display(XGB_parameters)

{'n_estimators': 150,
 'max_depth': 15,
 'min_samples_split': 10,
 'max_features': 'log2',
 'class_weight': 'balanced'}

{'objective': 'multi:softmax',
 'num_class': 3,
 'n_estimators': 150,
 'max_depth': 3,
 'learning_rate': 0.4,
 'verbosity': 0}

In [7]:
models = {
    "XGB": ignition_xbg, 
    "RF": ignition_rf
}

## SHAP

In [8]:
xgb_top_10 = get_shap(ignition_xbg, X_xgb, y_xgb)
xgb_top_10


,0,1
0,intermix_zone,0.386199
1,road_length_meters,0.376939
2,power_line_density,0.342932
3,Minimum Relative Humidity 7 Day Avg,0.321541
4,Energy Release Component,0.304266
5,Solar Radiation 7 Day Avg,0.300983
6,Wind Speed_x_100-hour Dead Fuel Moisture,0.285064
7,1000-hour Dead Fuel Moisture,0.276624
8,power_line_density_x_total_housing,0.256424
9,Wind Speed_x_elevation_range,0.238418


In [9]:
rf_top_10 = get_shap(ignition_rf, X_rf, y_rf)
rf_top_10 

 98%|===================| 881/900 [00:19<00:00]        

,0,1
0,road_length_meters,0.020613
1,power_line_density_x_total_housing,0.020027
2,population_density,0.019727
3,total_housing,0.017659
4,road_density,0.015517
5,total_population,0.014471
6,intermix_zone,0.012822
7,interface_zone,0.012372
8,influence_zone,0.012334
9,housing_density,0.011419


## Ablation

In [10]:
for set_name, columns in feature_sets.items():
    print(f"{set_name}: {columns}")
    print("\n")

Water Demand: ['Actual Evapotranspiration', 'Solar Radiation', 'Solar Radiation 7 Day Avg', 'Daily Minimum Air Temperature', 'Daily Maximum Air Temperature', 'Daily Maximum Air Temperature 7 Day Avg', 'Daily Minimum Air Temperature 7 Day Avg', 'Vapor Pressure Deficit', 'Vapor Pressure Deficit 7 Day Avg', 'Wind Speed', 'Wind Speed 7 Day Avg']


Water Supply: ['Precipitation', 'Precipitation 7 Day Avg', 'Maximum Relative Humidity', 'Minimum Relative Humidity', 'Maximum Relative Humidity 7 Day Avg', 'Minimum Relative Humidity 7 Day Avg', 'Specific Humidity', '100-hour Dead Fuel Moisture', '1000-hour Dead Fuel Moisture']


Water Supply Indexes: ['SPI 30-Day', 'SPI 180-Day', 'SPEI 30-Day', 'SPEI 90-Day', 'SPEI 180-Day', 'Palmer Drought Severity Index']


Fire Danger: ['Burning Index', 'Energy Release Component', 'Santa_Ana_Score']


Social: ['total_housing', 'total_population', 'housing_density', 'population_density', 'median_income']


Elevation: ['elevation_range', 'elevation_mean', 'elev

In [11]:
compare = []

compare.append(compare_model(ignition_xbg, X, y, best_strategy,
                                     name = 'XGB', test_set = 'Full'))

compare.append(compare_model(ignition_rf, X, y, best_strategy,
                                     name = 'RF', test_set = 'Full'))

for set_name, set_list in feature_sets.items():
    for model_name, model in models.items():
        print("Testing " + f"{model_name}: {set_name}")
        X_section = X.drop(columns = set_list)
        compare.append(compare_model(model, X_section, y, best_strategy,
                                     name = model_name, test_set = set_name))

Testing XGB: Water Demand
Testing RF: Water Demand
Testing XGB: Water Supply
Testing RF: Water Supply
Testing XGB: Water Supply Indexes
Testing RF: Water Supply Indexes
Testing XGB: Fire Danger
Testing RF: Fire Danger
Testing XGB: Social
Testing RF: Social
Testing XGB: Elevation
Testing RF: Elevation
Testing XGB: WUI
Testing RF: WUI
Testing XGB: Ecoregion
Testing RF: Ecoregion
Testing XGB: Land Cover
Testing RF: Land Cover
Testing XGB: Interactions
Testing RF: Interactions
Testing XGB: Wind Slope
Testing RF: Wind Slope
Testing XGB: Others
Testing RF: Others
Testing XGB: Coded Ecoregions
Testing RF: Coded Ecoregions
Testing XGB: Coded Seasons
Testing RF: Coded Seasons


In [12]:
comparisons = pd.DataFrame(compare)

In [14]:
XGB = comparisons[comparisons['Model'] == 'XGB'] 
RF = comparisons[comparisons['Model'] == 'RF'] 

In [18]:
full_XGB = XGB.loc[XGB['Test Set']=='Full','Macro F1'].values[0]
full_RF = RF.loc[RF['Test Set']=='Full','Macro F1'].values[0]

In [20]:
XGB['Macro F1 Percent Difference'] = ((full_XGB - XGB['Macro F1']) / full_XGB) * 100
RF['Macro F1 Percent Difference'] = ((full_RF - RF['Macro F1']) / full_RF) * 100

C:\Users\dusti\AppData\Local\Temp\ipykernel_7396\2524239291.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  XGB['Macro F1 Percent Difference'] = ((full_XGB - XGB['Macro F1']) / full_XGB) * 100
C:\Users\dusti\AppData\Local\Temp\ipykernel_7396\2524239291.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  RF['Macro F1 Percent Difference'] = ((full_RF - RF['Macro F1']) / full_RF) * 100


In [21]:
RF

,Test Set,Model,Weighted F1,Macro F1,High Risk Recall,Macro F1 Percent Difference
1,Full,RF,0.577962,0.586598,0.427273,0.000000
3,Water Demand,RF,0.616901,0.622928,0.490909,-6.193335
5,Water Supply,RF,0.604948,0.613284,0.463636,-4.549314
7,Water Supply Indexes,RF,0.581478,0.589815,0.436364,-0.548405
9,Fire Danger,RF,0.599213,0.606588,0.463636,-3.407907
11,Social,RF,0.598970,0.606718,0.454545,-3.429978
13,Elevation,RF,0.600557,0.607460,0.481818,-3.556537
15,WUI,RF,0.590421,0.597904,0.463636,-1.927380
17,Ecoregion,RF,0.593809,0.602437,0.436364,-2.700260
19,Land Cover,RF,0.590750,0.598489,0.436364,-2.027102


In [22]:
XGB

,Test Set,Model,Weighted F1,Macro F1,High Risk Recall,Macro F1 Percent Difference
0,Full,XGB,0.608247,0.611893,0.500000,0.000000
2,Water Demand,XGB,0.589140,0.597523,0.445455,2.348448
4,Water Supply,XGB,0.579275,0.589507,0.445455,3.658503
6,Water Supply Indexes,XGB,0.589902,0.592533,0.445455,3.163860
8,Fire Danger,XGB,0.583550,0.591348,0.454545,3.357523
10,Social,XGB,0.608080,0.613548,0.509091,-0.270499
12,Elevation,XGB,0.627455,0.634245,0.518182,-3.652988
14,WUI,XGB,0.600307,0.606759,0.463636,0.839007
16,Ecoregion,XGB,0.610946,0.617315,0.500000,-0.886103
18,Land Cover,XGB,0.605472,0.611105,0.500000,0.128686
